In [0]:
# Camada Bronze
# Nessa camada se cria os dados brutos com problemas propositais. Inclui duplicatas,
# nulos e negativos.

# Importa as bibliotecas necessárias
# pyspark é um "python" para big data
from pyspark.sql import SparkSession
from pyspark.sql.types import *

# SparkSession é a porta de entrada pro uso de PySpark
spark = SparkSession.builder.appName("pipeline_bsg").getOrCreate()

# Criação de dados brutos com alguns problemas
dados_brutos = [
    (1,  "Ana Silva",    "Eletronicos", 1500.00, "2024-01-15", "SP"),
    (2,  "Carlos Lima",  "Roupas",       250.00, "2024-01-16", "RJ"),
    (3,  "Juliana Costa","Eletronicos", 3200.00, "2024-01-17", "MG"),
    (4,  "Pedro Alves",  "Alimentos",    180.00, "2024-01-18", "SP"),
    (5,  "Mariana Reis", "Roupas",       420.00, "2024-01-19", "RS"),
    (2,  "Carlos Lima",  "Roupas",       250.00, "2024-01-16", "RJ"),  # duplicata proposital
    (6,  None,           "Eletronicos",  980.00, "2024-01-20", "SP"),  # nome nulo proposital
    (7,  "Rafael Souza", None,           150.00, "2024-01-21", "RJ"),  # categoria nula proposital
    (8,  "Tatiane Reis", "Alimentos",   -50.00,  "2024-01-22", "MG"),  # valor negativo proposital
    (9,  "Bruno Costa",  "Moveis",      2800.00, "2024-01-23", "SP"),
    (10, "Luciana Ferr", "Eletronicos", 1100.00, "2024-01-24", "RS"),
]

# Definindo o esquema e tipo de dado de cada coluna

schema = StructType([
    StructField("id",   IntegerType(), True),
    StructField("cliente", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("valor", DoubleType(), True),
    StructField("data", StringType(), True),
    StructField("estado", StringType(), True),
])

# Criando o DataFrame - tabela em memória com os dados brutos
# DataFrame é o equivalente a um dataframe do pandas

df_bronze = spark.createDataFrame(dados_brutos, schema)

# Salvando como Delta Table na camada bronze
# Delta Table é um formato de arquivo do Databricks

df_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_vendas")

# Exibe os dados para conferir

print(f"Bronze: {df_bronze.count()} registros salvos - incluindo problemas propositais")
df_bronze.show()

Bronze: 11 registros salvos - incluindo problemas propositais
+---+-------------+-----------+------+----------+------+
| id|      cliente|  categoria| valor|      data|estado|
+---+-------------+-----------+------+----------+------+
|  1|    Ana Silva|Eletronicos|1500.0|2024-01-15|    SP|
|  2|  Carlos Lima|     Roupas| 250.0|2024-01-16|    RJ|
|  3|Juliana Costa|Eletronicos|3200.0|2024-01-17|    MG|
|  4|  Pedro Alves|  Alimentos| 180.0|2024-01-18|    SP|
|  5| Mariana Reis|     Roupas| 420.0|2024-01-19|    RS|
|  2|  Carlos Lima|     Roupas| 250.0|2024-01-16|    RJ|
|  6|         NULL|Eletronicos| 980.0|2024-01-20|    SP|
|  7| Rafael Souza|       NULL| 150.0|2024-01-21|    RJ|
|  8| Tatiane Reis|  Alimentos| -50.0|2024-01-22|    MG|
|  9|  Bruno Costa|     Moveis|2800.0|2024-01-23|    SP|
| 10| Luciana Ferr|Eletronicos|1100.0|2024-01-24|    RS|
+---+-------------+-----------+------+----------+------+



In [0]:
# Camada Silver
# Aqui limpamos os problemas da camada Bronze
# Removemos duplicatas, nulos e valores inválidos
# e criamos colunas novas com informações úteis

# Importando funções do PySpark para transformar dados
# col = acessa uma coluna, when = condicional tipo if/else
# month/year = extrai mês e ano de uma data
from pyspark.sql.functions import col, when, month, year, to_date

# Lendo os dados da camada Bronze que salvamos antes
df_silver = spark.table("bronze_vendas")

# Removendo duplicatas — linhas completamente iguais
df_silver = df_silver.dropDuplicates()

# Removendo linhas com nulos nas colunas obrigatórias
df_silver = df_silver.dropna(subset=["cliente", "categoria"])

# Removendo valores negativos — venda não pode ser negativa
df_silver = df_silver.filter(col("valor") > 0)

# Convertendo a coluna data de texto para tipo Date
df_silver = df_silver.withColumn("data", to_date(col("data"), "yyyy-MM-dd"))

# Criando coluna de mês e ano para facilitar análises
df_silver = df_silver.withColumn("mes",  month(col("data")))
df_silver = df_silver.withColumn("ano",  year(col("data")))

# Criando coluna de região com base no estado
# when é uma condição como um if/else — se  estado for SP, RJ, MG ou ES então se torna Sudeste
df_silver = df_silver.withColumn("regiao",
    when(col("estado").isin("SP", "RJ", "MG", "ES"), "Sudeste")
    .when(col("estado").isin("RS", "SC", "PR"),       "Sul")
    .otherwise("Outros"))

# Salvando como Delta Table na camada Silver
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_vendas")

print(f"Silver: {df_silver.count()} registros limpos")
print("Removidos: 1 duplicata + 2 nulos + 1 negativo = 4 registros")
df_silver.show()


Silver: 7 registros limpos
Removidos: 1 duplicata + 2 nulos + 1 negativo = 4 registros
+---+-------------+-----------+------+----------+------+---+----+-------+
| id|      cliente|  categoria| valor|      data|estado|mes| ano| regiao|
+---+-------------+-----------+------+----------+------+---+----+-------+
|  1|    Ana Silva|Eletronicos|1500.0|2024-01-15|    SP|  1|2024|Sudeste|
|  2|  Carlos Lima|     Roupas| 250.0|2024-01-16|    RJ|  1|2024|Sudeste|
|  3|Juliana Costa|Eletronicos|3200.0|2024-01-17|    MG|  1|2024|Sudeste|
|  4|  Pedro Alves|  Alimentos| 180.0|2024-01-18|    SP|  1|2024|Sudeste|
|  5| Mariana Reis|     Roupas| 420.0|2024-01-19|    RS|  1|2024|    Sul|
|  9|  Bruno Costa|     Moveis|2800.0|2024-01-23|    SP|  1|2024|Sudeste|
| 10| Luciana Ferr|Eletronicos|1100.0|2024-01-24|    RS|  1|2024|    Sul|
+---+-------------+-----------+------+----------+------+---+----+-------+



In [0]:
# Camada Gold
# Aqui calculamos os indicadores finais
# Os dados estão agregados e prontos para análise

# Importando funções de agregação
# sum = soma, avg = média, count = contagem
# round = arredonda, desc = ordem decrescente
from pyspark.sql.functions import sum, avg, count, round, desc

# Lendo os dados limpos da camada Silver
df_silver = spark.table("silver_vendas")

# Agregação 1 — Faturamento por categoria
# groupBy agrupa os dados igual ao GROUP BY do SQL
gold_categoria = df_silver.groupBy("categoria").agg(
    count("id").alias("total_vendas"),          # quantas vendas por categoria
    round(sum("valor"), 2).alias("faturamento"), # soma dos valores
    round(avg("valor"), 2).alias("ticket_medio") # valor médio por venda
).orderBy(desc("faturamento"))                   # ordena do maior para o menor

# Agregação 2 — Faturamento por região
gold_regiao = df_silver.groupBy("regiao").agg(
    count("id").alias("total_vendas"),
    round(sum("valor"), 2).alias("faturamento")
).orderBy(desc("faturamento"))

# Salvando as duas tabelas Gold
gold_categoria.write.format("delta").mode("overwrite").saveAsTable("gold_categoria")
gold_regiao.write.format("delta").mode("overwrite").saveAsTable("gold_regiao")

print("=== GOLD — Faturamento por Categoria ===")
gold_categoria.show()

print("=== GOLD — Faturamento por Região ===")
gold_regiao.show()

=== GOLD — Faturamento por Categoria ===
+-----------+------------+-----------+------------+
|  categoria|total_vendas|faturamento|ticket_medio|
+-----------+------------+-----------+------------+
|Eletronicos|           3|     5800.0|     1933.33|
|     Moveis|           1|     2800.0|      2800.0|
|     Roupas|           2|      670.0|       335.0|
|  Alimentos|           1|      180.0|       180.0|
+-----------+------------+-----------+------------+

=== GOLD — Faturamento por Região ===
+-------+------------+-----------+
| regiao|total_vendas|faturamento|
+-------+------------+-----------+
|Sudeste|           5|     7930.0|
|    Sul|           2|     1520.0|
+-------+------------+-----------+



In [0]:
# Validação do pipeline
# Aqui confirmamos que cada camada tem os dados corretos
# e que o pipeline funcionou

# Lendo as 4 tabelas criadas no pipeline
bronze     = spark.table("bronze_vendas")
silver     = spark.table("silver_vendas")
gold_cat   = spark.table("gold_categoria")
gold_reg   = spark.table("gold_regiao")

# Exibindo o resumo de cada camada
print("=== RESUMO DO PIPELINE ===")
print(f"Bronze:         {bronze.count()} registros — dados brutos com problemas")
print(f"Silver:         {silver.count()} registros — dados limpos e tratados")
print(f"Gold Categoria: {gold_cat.count()} categorias agregadas")
print(f"Gold Regiao:    {gold_reg.count()} regioes agregadas")
print("")
print("=== REGISTROS REMOVIDOS NA SILVER ===")
print(f"Removidos: {bronze.count() - silver.count()} registros")
print("  - 1 duplicata")
print("  - 2 registros com nulos")
print("  - 1 registro com valor negativo")
print("")
print("Pipeline Bronze -> Silver -> Gold concluido com sucesso!")

=== RESUMO DO PIPELINE ===
Bronze:         11 registros — dados brutos com problemas
Silver:         7 registros — dados limpos e tratados
Gold Categoria: 4 categorias agregadas
Gold Regiao:    2 regioes agregadas

=== REGISTROS REMOVIDOS NA SILVER ===
Removidos: 4 registros
  - 1 duplicata
  - 2 registros com nulos
  - 1 registro com valor negativo

Pipeline Bronze -> Silver -> Gold concluido com sucesso!


In [0]:
# Preparando os dados para visualização em dashboard

# Lendo as tabelas Gold que criamos
gold_cat = spark.table("gold_categoria")
gold_reg = spark.table("gold_regiao")

# Exibindo as tabelas de forma organizada
display(gold_cat)

categoria,total_vendas,faturamento,ticket_medio
Eletronicos,3,5800.0,1933.33
Moveis,1,2800.0,2800.0
Roupas,2,670.0,335.0
Alimentos,1,180.0,180.0


Databricks visualization. Run in Databricks to view.

In [0]:
# Exibindo faturamento por região
display(gold_reg)

regiao,total_vendas,faturamento
Sudeste,5,7930.0
Sul,2,1520.0


Databricks visualization. Run in Databricks to view.